# Erde-Mond-Simulation

Dieses Notebook enthaelt den kompletten Code aus `src/` (Body, Constants, Integrator, Simulation, Scenarios, Visualization) als eigene Module, sodass es ohne die Projektdateien laeuft. Tests sind nicht enthalten.

In [16]:
import sys, types

%matplotlib tk
import matplotlib.pyplot as plt

def load_module(name, source):
    """Fuehrt den Quellcode einer Datei als eigenstaendiges Modul aus, damit 'import <name>' funktioniert."""
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(compile(source, f"{name}.py", "exec"), module.__dict__)
    return module


## Constants (`src/Constants.py`)

In [17]:
Constants = load_module("Constants", r'''
"""

UNIT CONVENTION (SI)
    Length      in meters        [m]
    Mass        in kilograms      [kg]
    Time        in seconds        [s]
    Velocity    in [m/s]
    Angle       internally in degrees [deg]  (convert to rad only when computing)

"""

import math

# ---------------------------------------------------------------------------
# 1. Fundamental constant
# ---------------------------------------------------------------------------

G = 6.67430e-11


# ---------------------------------------------------------------------------
# 2. Earth
# ---------------------------------------------------------------------------

EARTH_MASS   = 5.972e24
EARTH_RADIUS = 6.371e6


# ---------------------------------------------------------------------------
# 3. Moon
# ---------------------------------------------------------------------------

MOON_MASS   = 7.346e22
MOON_RADIUS = 1.7374e6


# Orbit of the Moon around Earth (center to center):
EARTH_MOON_DISTANCE = 3.844e8
MOON_ORBITAL_SPEED  = 1022.0
MOON_ORBITAL_PERIOD = 27.322 * 86400.0




# Initial state for the simulation (Earth sits in the origin 0/0/0):
#   the Moon starts on the +X axis and moves in the +Y direction,
#   so it orbits inside the X/Y plane
MOON_START_X  = EARTH_MOON_DISTANCE   # start position on the X axis [m]
MOON_START_VY = MOON_ORBITAL_SPEED    # start velocity in the Y direction [m/s]


# ---------------------------------------------------------------------------
# 4. Derived reference values (to validate the simulation)
# ---------------------------------------------------------------------------

# Earth escape velocity:  v = sqrt(2 * G * M / R)   -> approx. 11,186 m/s
EARTH_ESCAPE_VELOCITY = math.sqrt(2.0 * G * EARTH_MASS / EARTH_RADIUS)

# Theoretical circular orbital velocity of the Moon:  v = sqrt(G * M / r)
# -> approx. 1018 m/s, should match MOON_ORBITAL_SPEED (1022) -> sanity check
MOON_CIRCULAR_VELOCITY = math.sqrt(G * EARTH_MASS / EARTH_MOON_DISTANCE)


# ---------------------------------------------------------------------------
# 5. Time units (for readable time steps and time-lapse)
# ---------------------------------------------------------------------------

SECOND = 1.0
MINUTE = 60.0 * SECOND
HOUR   = 60.0 * MINUTE
DAY    = 24.0 * HOUR             # 86,400 s

DEFAULT_TIME_STEP = 60.0         # default time step [s] = 1 minute per step


# ---------------------------------------------------------------------------
# 6. Length unit (only for display conversion)
# ---------------------------------------------------------------------------

KM = 1000.0                      # 1 kilometer = 1000 meters


# ---------------------------------------------------------------------------
# 7. Task 2.1 - The Gun Club and the Columbiad
# ---------------------------------------------------------------------------

# Given launch velocities [m/s] (PDF: 7, 9, 12 km/s):
CANNON_SPEEDS = (7.0 * KM, 9.0 * KM, 12.0 * KM)   # = (7000, 9000, 12000) m/s

# Lead angles [degrees]: where the gun points relative to the Moon
#   0 deg  = exactly at the Moon
#   15 deg = 15 degrees ahead of the Moon's position
#   30 deg = 30 degrees ahead of the Moon's position
CANNON_LEAD_ANGLES = (0.0, 15.0, 30.0)

# Projectile properties (free to choose - tiny compared to Earth/Moon):
PROJECTILE_MASS   = 1.0e4        # [kg]  = 10 tons (example value)
PROJECTILE_RADIUS = 1.0          # [m]   (point mass, only for collision test)
'''
)

## Body (`src/Body.py`)

In [18]:
Body = load_module("Body", r'''
import numpy as np


class Body:


    name: str
    mass: float
    radius: float
    position: np.ndarray
    velocity: np.ndarray
    position_previous: np.ndarray
    acceleration: np.ndarray


    def __init__(self, name, mass, radius, position, velocity):

        if mass <= 0:
            raise ValueError(f"mass must be positive, got {mass}")
        if radius <= 0:
            raise ValueError(f"radius must be positive, got {radius}")

        self.name = name
        self.mass = mass
        self.radius = radius

        # in numpy array umwandeln
        self.position = np.array(position, dtype=float)
        self.velocity = np.array(velocity, dtype=float)

        # muss immer Achsen
        if self.position.shape != (3,):
            raise ValueError(f"position must have 3 values [x, y, z], got {position}")
        if self.velocity.shape != (3,):
            raise ValueError(f"velocity must have 3 values [vx, vy, vz], got {velocity}")


        self.position_previous = None
        self.acceleration = np.array([0.0, 0.0, 0.0])



    def diameter(self):
        return 2.0 * self.radius

    def distance_to(self, other_body):
        return np.linalg.norm(other_body.position - self.position)

    def distance_vector_to(self, other_body):
       return other_body.position - self.position

    def is_touching(self, other_body):
        return self.distance_to(other_body) <= self.radius + other_body.radius

    def momentum(self):
        return self.mass * self.velocity

    def kinetic_energy(self):
        return 0.5 * self.mass * np.linalg.norm(self.velocity)**2



    def __repr__(self):
        return f"Body('{self.name}', mass={self.mass:.3e} kg, position={self.position})"

'''
)

## Integrator (`src/Integrator.py`)

In [19]:
Integrator = load_module("Integrator", r'''
import numpy as np
from abc import ABC, abstractmethod
from Constants import G
from Body import Body



class Integrator(ABC):
    @abstractmethod
    def step(self, bodies, dt):
        pass


    def calculate_acceleration(self, bodies):
        for i in bodies:
            i.acceleration = np.zeros(3)
        for i in range(len(bodies)):
            for j in range(i + 1, len(bodies)):

                p1 = bodies[i]
                p2 = bodies[j]
                direction = Body.distance_vector_to(p1,p2)
                r = np.linalg.norm(direction)
                if r > 1e-12:
                    direction_norm = direction / r

                    F = G * p1.mass * p2.mass / r**2

                    a1 = F / p1.mass
                    a2 = F / p2.mass

                    bodies[i].acceleration += a1 * direction_norm
                    bodies[j].acceleration -= a2 * direction_norm

    def potential_energy(self, bodies):
        E_pot = 0.0
        for i in range(len(bodies)):
            for j in range(i + 1, len(bodies)):
                r = np.linalg.norm(bodies[j].position - bodies[i].position)
                if r > 1e-12:
                    E_pot += -G * bodies[i].mass * bodies[j].mass / r
        return E_pot





class  Verlet(Integrator):
    def step(self, bodies, dt):

        Integrator.calculate_acceleration(self, bodies)

        # Mini-Euler only at the first step:
        # sets position_previous backwards, without moving the position
        for body in bodies:
            if body.position_previous is None:
                body.position_previous = body.position - body.velocity * dt

        for i in range(len(bodies)):
            prev_pos = bodies[i].position_previous.copy()
            temp_pos = bodies[i].position.copy()

            bodies[i].position = bodies[i].position * 2 - prev_pos + bodies[i].acceleration.copy() * dt * dt
            bodies[i].position_previous = temp_pos

            # Calculate velocity implicitly from position change
            bodies[i].velocity = (bodies[i].position.copy() - prev_pos) / (2.0 * dt)




class Euler(Integrator):

    def step(self, bodies, dt):
        # explicit euler with big errors
        Integrator.calculate_acceleration(self, bodies)

        for body in bodies:
            temp_vel = body.velocity.copy()

            body.velocity += body.acceleration * dt
            body.position += temp_vel * dt
'''
)

## Simulation (`src/Simulation.py`)

In [20]:
Simulation = load_module("Simulation", r'''
from Integrator import Verlet, Euler

class Simulation:

    def __init__(self, bodies, config, integrator=None):
        self.bodies = bodies
        self.dt = config["time_step"]
        self.integrator = integrator if integrator is not None else Verlet()
        self.t = 0.0
        self.history = []
        self._save_snapshot()
    
    def step(self):
        self.integrator.step(self.bodies, self.dt)
        # Check here for collisions, when implemented
        self.t += self.dt
        self._save_snapshot()

    def _save_snapshot(self):
        snapshot = {
            "t": self.t,
            "bodies": [
                {
                "name": b.name,
                "mass": b.mass,
                "position": b.position.copy(),
                "velocity": b.velocity.copy()
                }
                for b in self.bodies
            ]
        }
        self.history.append(snapshot)

    def simulate(self, steps):
        for i in range(steps):
            self.step()
'''
)

## Scenarios (`src/Scenarios.py`)

In [21]:
Scenarios = load_module("Scenarios", r'''
import math
from Body import Body
import Constants


def create_earth_moon():
    earth = Body("Earth",Constants.EARTH_MASS,Constants.EARTH_RADIUS,[0,0,0],[0,0,0])
    moon = Body("Moon",Constants.MOON_MASS,Constants.MOON_RADIUS,[Constants.MOON_START_X,0,0],[0,Constants.MOON_CIRCULAR_VELOCITY,0])
    config = {"time_step": Constants.DEFAULT_TIME_STEP}
    return [earth,moon],config

def create_cannon_shot(speed,angle_deg):
    angle_rad = math.radians(angle_deg)
    earth = Body("Earth",Constants.EARTH_MASS,Constants.EARTH_RADIUS,[0,0,0],[0,0,0])
    moon = Body("Moon",Constants.MOON_MASS,Constants.MOON_RADIUS,[Constants.MOON_START_X,0,0],[0,Constants.MOON_CIRCULAR_VELOCITY,0])
    pos_x = Constants.EARTH_RADIUS * math.cos(angle_rad)
    pos_y = Constants.EARTH_RADIUS * math.sin(angle_rad)
    vel_x = speed * math.cos(angle_rad)
    vel_y = speed * math.sin(angle_rad)
    projectile = Body("Projectile",Constants.PROJECTILE_MASS,Constants.PROJECTILE_RADIUS,[pos_x,pos_y,0],[vel_x,vel_y,0])
    config = {"time_step": Constants.SECOND}
    return [earth,moon,projectile],config
'''
)

## Visualization (`src/Visualization.py`)

In [22]:
Visualization = load_module("Visualization", r'''
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
from matplotlib.widgets import RadioButtons, TextBox, Button

import Constants
from Integrator import Verlet, Euler
from Simulation import Simulation
from Scenarios import create_earth_moon, create_cannon_shot

MIN_DISPLAY_FRACTION = 0.012


def extract_trajectories(history):
    """Liest aus der Simulationshistorie für jeden Körper die Positionen über die Zeit aus."""
    if not history:
        raise ValueError("history must contain at least one snapshot")

    names = [body_snapshot["name"] for body_snapshot in history[0]["bodies"]]
    times = np.array([snapshot["t"] for snapshot in history])
    positions = {
        name: np.array([snapshot["bodies"][index]["position"] for snapshot in history])
        for index, name in enumerate(names)
    }
    return times, positions


def _planar_extent(bodies, positions):
    """Berechnet die größte Ausdehnung aller Körper in der X/Y-Ebene."""
    xy = np.vstack([positions[body.name][:, :2] for body in bodies])
    return max(xy[:, 0].max() - xy[:, 0].min(), xy[:, 1].max() - xy[:, 1].min())


def auto_body_scale(bodies, positions, visible_fraction=0.05):
    """Berechnet einen Vergrößerungsfaktor, sodass der größte Körper sichtbar bleibt."""
    max_radius = max(body.radius for body in bodies)
    extent = _planar_extent(bodies, positions)
    return (visible_fraction * extent) / max_radius


def animate_system(bodies, history, unit=Constants.KM, unit_label="km",
                    frame_step=1, trail_length=200, interval=30, figsize=(5, 5), dpi=80, title=None):
    """Erstellt eine Animation, die die Simulation Schritt für Schritt abspielt."""
    times, positions = extract_trajectories(history)
    frame_indices = list(range(0, len(times), frame_step))

    scale = auto_body_scale(bodies, positions)
    palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    all_xy = np.vstack([positions[body.name][:, :2] for body in bodies]) / unit
    span = max(all_xy[:, 0].max() - all_xy[:, 0].min(), all_xy[:, 1].max() - all_xy[:, 1].min())
    margin = 0.1 * span if span > 0 else 1.0
    ax.set_xlim(all_xy[:, 0].min() - margin, all_xy[:, 0].max() + margin)
    ax.set_ylim(all_xy[:, 1].min() - margin, all_xy[:, 1].max() + margin)
    ax.set_aspect("equal")
    ax.set_xlabel(f"x [{unit_label}]")
    ax.set_ylabel(f"y [{unit_label}]")
    ax.set_title(title or "Simulation")

    trails = {}
    markers = {}
    for index, body in enumerate(bodies):
        color = palette[index % len(palette)]
        trails[body.name], = ax.plot([], [], linewidth=1, alpha=0.5, color=color)
        # Mindestgröße MIN_DISPLAY_FRACTION * span, damit auch sehr kleine Körper (z.B. das Geschoss) sichtbar bleiben
        display_radius = max(body.radius * scale / unit, MIN_DISPLAY_FRACTION * span)
        circle = Circle((0.0, 0.0), display_radius, zorder=3,
                        facecolor=color, edgecolor="black", linewidth=0.6, label=body.name)
        ax.add_patch(circle)
        markers[body.name] = circle
    ax.legend(loc="upper right", fontsize="small")
    time_label = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top")

    def update(frame_number):
        # Bewegt Marker und Bahnspuren zum jeweils nächsten aufgezeichneten Zeitschritt
        index = frame_indices[frame_number]
        trail_start = max(0, index - trail_length)
        for body in bodies:
            xy = positions[body.name][:, :2] / unit
            markers[body.name].center = xy[index]
            trails[body.name].set_data(xy[trail_start:index + 1, 0], xy[trail_start:index + 1, 1])
        time_label.set_text(f"t = {times[index] / Constants.DAY:.2f} d")
        return [*markers.values(), *trails.values(), time_label]

    return animation.FuncAnimation(fig, update, frames=len(frame_indices), interval=interval, blit=True)


def plot_distance_over_time(history, name_a, name_b, collision_distance=None,
                             unit=Constants.KM, unit_label="km", ax=None, title=None):
    """Zeichnet den Abstand zwischen zwei Körpern über die Zeit."""
    times, positions = extract_trajectories(history)
    if name_a not in positions:
        raise KeyError(f"no body named '{name_a}' in this history")
    if name_b not in positions:
        raise KeyError(f"no body named '{name_b}' in this history")

    distance = np.linalg.norm(positions[name_a] - positions[name_b], axis=1)

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))

    ax.plot(times / Constants.DAY, distance / unit, label=f"Abstand {name_a} - {name_b}")
    if collision_distance is not None:
        ax.axhline(collision_distance / unit, color="red", linestyle="--",
                   label="Kontaktabstand (Kollision)")

    ax.set_xlabel("Zeit [Tage]")
    ax.set_ylabel(f"Abstand [{unit_label}]")
    ax.set_title(title or f"Abstand {name_a} - {name_b} über die Zeit")
    ax.legend(loc="best", fontsize="small")
    return ax


# ---------------------------------------------------------------------------
# Demo: Zeigt zuerst das Erde-Mond-System
# (Aufgabe 1) und öffnet dann ein Fenster, in dem Integrator, Geschwindigkeit und
# Vorhaltewinkel für einen Kanonenschuss (Aufgabe 2.1) eingestellt werden können.
# Jeder Klick auf "Schuss starten" simuliert und animiert einen weiteren Schuss.
# ---------------------------------------------------------------------------

# hält Referenzen auf alle Animationen, damit sie nicht vom Garbage Collector entfernt werden
_active_animations = []


def _frame_step_for(history, target_frames=150):
    """Wählt frame_step so, dass eine Animation über `history` ungefähr `target_frames` Frames hat."""
    return max(1, len(history) // target_frames)


def _show_earth_moon_orbit(orbits=1):
    """Aufgabe 1: simuliert `orbits` Mondumläufe und zeigt sie als Animation mit vollständiger Bahnspur."""
    bodies, config = create_earth_moon()
    earth, _moon = bodies
    sim = Simulation(bodies, config, Verlet())
    orbit_steps = int(orbits * Constants.MOON_ORBITAL_PERIOD / config["time_step"])
    sim.simulate(orbit_steps)

    orbit_label = "ein voller Orbit" if orbits == 1 else f"{orbits} Orbits"
    _active_animations.append(animate_system(bodies, sim.history, frame_step=_frame_step_for(sim.history),
                                               trail_length=len(sim.history),
                                               title=f"Der Mond umkreist die Erde ({orbit_label})"))
    # Zweite Animation, gezoomt auf die kleine Eigenbewegung der Erde um den Schwerpunkt
    _active_animations.append(animate_system([earth], sim.history, frame_step=_frame_step_for(sim.history),
                                               trail_length=len(sim.history),
                                               title="Eigenbewegung der Erde um den Schwerpunkt"))
    plt.show(block=False)


def _run_cannon_shot(integrator, speed, lead_angle):
    """Aufgabe 2.1: simuliert einen Kanonenschuss und zeigt ihn als Animation sowie den Abstand zum Mond über die Zeit."""
    bodies, config = create_cannon_shot(speed, lead_angle)
    sim = Simulation(bodies, config, integrator)
    earth, moon, projectile = bodies

    # bis zu 30 Stunden simulieren, früher abbrechen sobald das Geschoss Erde oder Mond berührt
    max_steps = int(30 * Constants.HOUR / config["time_step"])
    for _ in range(max_steps):
        sim.step()
        if projectile.is_touching(earth) or projectile.is_touching(moon):
            break

    title = (f"Kanonenschuss: v0 = {speed:.0f} m/s, Vorhaltewinkel = {lead_angle:.1f} deg, "
             f"{type(integrator).__name__} (Flugzeit {sim.t / Constants.HOUR:.2f} h)")
    _active_animations.append(animate_system(sim.bodies, sim.history,
                                               frame_step=_frame_step_for(sim.history, target_frames=100),
                                               trail_length=len(sim.history), title=title))
    plot_distance_over_time(sim.history, "Projectile", "Moon",
                            collision_distance=projectile.radius + moon.radius,
                            title="Abstand Geschoss-Mond")
    plt.show(block=False)


def _open_cannon_shot_launcher():
    """Öffnet ein Fenster zur Auswahl von Integrator, Geschwindigkeit und Vorhaltewinkel; jeder Klick auf
    "Schuss starten" simuliert und animiert einen weiteren Kanonenschuss mit den eingestellten Werten."""
    fig = plt.figure(figsize=(4, 3))
    fig.suptitle("Kanonenschuss")

    integrator_radio = RadioButtons(fig.add_axes((0.1, 0.45, 0.35, 0.4)), ("Verlet", "Euler"))
    speed_box = TextBox(fig.add_axes((0.55, 0.7, 0.35, 0.1)), "v0 [m/s]", initial="12000")
    angle_box = TextBox(fig.add_axes((0.55, 0.5, 0.35, 0.1)), "Winkel [deg]", initial="0")
    shoot_button = Button(fig.add_axes((0.55, 0.25, 0.35, 0.15)), "Schuss starten")

    def on_shoot_clicked(_event):
        # liest die aktuellen Einstellungen aus und startet damit einen weiteren Kanonenschuss
        integrator = Verlet() if integrator_radio.value_selected == "Verlet" else Euler()
        _run_cannon_shot(integrator, float(speed_box.text), float(angle_box.text))

    shoot_button.on_clicked(on_shoot_clicked)
    plt.show()


def main():
    """Zeigt das Erde-Mond-System und öffnet danach den Kanonenschuss-Launcher für beliebig viele Schüsse."""
    _show_earth_moon_orbit(2)
    _open_cannon_shot_launcher()


if __name__ == "__main__":
    main()

'''
)

## Aufgabe 1: Erde-Mond-System

Zeigt 2 Mondumlaeufe und zoomt zusaetzlich auf die Eigenbewegung der Erde um den Schwerpunkt.

In [23]:
Visualization._show_earth_moon_orbit(orbits=2)

## Aufgabe 2.1: Kanonenschuss-Launcher

Integrator, Geschwindigkeit und Vorhaltewinkel waehlen und beliebig oft schiessen.

In [24]:
Visualization._open_cannon_shot_launcher()